In [ ]:
import pandas as pd
data = pd.read_csv('/kaggle/input/amazon-echo-dot-2-reviews-dataset/Amazon Echo 2 Reviews.csv')
data.head()

In [ ]:
data_info = data.info()
data_description = data.describe()
data_info, data_description

In [ ]:
data_cleaned = data.drop(columns=['Declaration Text'])
data_cleaned['Review Text'].fillna("No Review", inplace=True)
data_cleaned['User Verified'].fillna("Unknown", inplace=True)
missing_values_summary = data_cleaned.isnull().sum()
missing_values_summary

In [ ]:
data_cleaned = data_cleaned.drop(columns=['Review Useful Count'])
remaining_columns = data_cleaned.columns
data_cleaned.info(), remaining_columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
plt.figure(figsize=(8, 6))
sns.countplot(x=data_cleaned['Rating'], palette="viridis")
plt.title("Distribution of Ratings", fontsize=16)
plt.xlabel("Rating", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer

In [ ]:
X = data_cleaned[['Review Text', 'Review Color', 'User Verified']]  # Features
y = data_cleaned['Rating']  # Target

pipeline = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(stop_words='english'), 'Review Text'),
            ('cat', OneHotEncoder(), ['Review Color', 'User Verified'])
        ]
    )),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
rmse

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['auto', 'sqrt', 'log2']
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Parameters:", best_params)
print("Best Cross-Validation Score:", best_score)